In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "UseBSForNot/MergeInsthData_1211.csv"))
dim(data)

[1] 3227892      78

In [4]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "fac_pub"                                     
 [4] "year"                                        
 [5] "paper_type"                                  
 [6] "paper_language"                              
 [7] "nwbin"                                       
 [8] "new_word_reuse_times"                        
 [9] "npbin"                                       
[10] "new_phrase_reuse_times"                      
[11] "novelbin"                                    
[12] "rao_stirling"                                
[13] "author_id"                                   
[14] "author_position"                             
[15] "institution_id"                              
[16] "country_code"                                
[17] "country"                                     
[18] "num_author_log"                              
[19] "num_inst_log"                                
[20] "num_co

In [5]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                                     fac_pub 
                                           0 
                                        year 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                       nwbin 
                                           0 
                        new_word_reuse_times 
                                           0 
                                       npbin 
                                           0 
                      new_phrase_reuse_times 
                                           0 
                                    novelbin 
                                           0 
                                rao_stirling 
                                      303430 
                                   author_id 
                                           0 
                             author_position 
                                           0 
                              institution_id 
                                           0 
                                country_code 
                                           0 
                                     country 
                                           0 
                              num_author_log 
                                           0 
                                num_inst_log 
                                           0 
                             num_country_log 
                                           0 
                           num_reference_log 
                                           0 
                             timescited5_log 
                                      107147 
                            timescited10_log 
                                      163886 
                           timescitedall_log 
                                           0 
                               ab_length_log 
                                           0 
                     leadership_global_class 
                                           0 
                         mean_career_age_log 
                                           0 
                            inst_h_index_log 
                                           6 
                                   source_id 
                                           0 
                          source_h_index_log 
                                           0 
                                 source_type 
                                           0 
                                 core_source 
                                           0 
        Agricultural.and.Biological.Sciences 
                                           0 
                         Arts.and.Humanities 
                                           0 
Biochemistry..Genetics.and.Molecular.Biology 
                                           0 
         Business..Management.and.Accounting 
                                           0 
                        Chemical.Engineering 
                                           0 
                                   Chemistry 
                                           0 
                            Computer.Science 
                                           0 
                           Decision.Sciences 
                                           0 
                                   Dentistry 
                                           0 
                Earth.and.Planetary.Sciences 
                                           0 
         Economics..Econometrics.and.Finance 
                                           0 
                                      Energy 
                                         

In [6]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [7]:
data <- data[!is.na(data$Atyp_10pct_Z), ]
dim(data)
data <- data[data$num_reference >=5, ]
dim(data)

[1] 2814103      78

[1] 2760385      78

In [8]:
# data <- data[data$ab_length > 0, ]
# dim(data)
data <- data[data$year >= 1950, ]
dim(data)

[1] 2760385      78

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$leadership_global_class <- factor(data$leadership_global_class)
data <- within(data, leadership_global_class <- relevel(leadership_global_class, ref = 'shared'))
data$fac_pub <- factor(data$fac_pub)
data <- within(data, fac_pub <- relevel(fac_pub, ref = 'NonBSF'))
data$core_source <- factor(data$core_source)
data <- within(data, core_source <- relevel(core_source, ref = 'NonCore'))
data$year <- as.factor(data$year)

In [11]:
# variable type
paper_level <- "num_author_log + num_inst_log + num_country_log + num_reference_log + leadership_global_class + mean_career_age_log + inst_h_index_log" 
journal_level <- "source_h_index_log + core_source"
disciplines <- "Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [12]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_ps <- feglm(fml, data = data[(data$domain_Physical.Sciences == 1), ], family = binomial("logit"), vcov = "iid")
summary(model_ps)

NOTES: 3 observations removed because of NA values (RHS: 3).
       20,215/0 fixed-effects (78,545 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 2,236,396
Fixed-effects: author_id: 55,424,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error    z value
fac_pubBSF                                    0.074731   0.006671  11.202080
num_author_log                                0.211550   0.005207  40.626516
num_inst_log                                 -0.003943   0.006613  -0.596263
num_country_log                              -0.293126   0.009664 -30.333097
num_reference_log                             0.146196   0.003269  44.724004
leadership_global_classallNorth               0.138545   0.009386  14.760193
leadership_global_classallSouth              -0.053631   0.015025  -3.569560
mean_career_age_log                           0.004271   0.005411   0.789263
inst_h_index_log                             -0.016366   0.004186  -3.909818
source_h_index_log                           -0.028386   0.002448 -11.593961
core_sou

In [13]:
paper_vars <- c("num_author_log", "num_inst_log", "num_country_log",
                "num_reference_log", "leadership_global_class",
                "mean_career_age_log", "inst_h_index_log" )


journal_vars <- c("source_h_index_log", "core_source")

disciplines <- c("Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

In [14]:
MEs = ggemmeans(model_ps, terms=c('fac_pub'), typical='median')

In [15]:
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.2747726,0.03536667,0.2611773,0.2887991,1
BSF,0.2899123,0.03536021,0.2758553,0.3043845,1


In [16]:
fname = paste0(main_path, "UseBSForNot/predict_ns_1211_ps.csv")
write.csv(MEs, fname, row.names = FALSE)

In [17]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_ls <- feglm(fml, data = data[(data$domain_Life.Sciences == 1), ], family = binomial("logit"), vcov = "iid")
summary(model_ls)

Warning message in formula.character(object, env = baseenv()):
“Using formula(x) is deprecated when x is a character vector of length > 1.
  Consider formula(paste(x, collapse = " ")) instead.”
NOTE: 16,802/0 fixed-effects (44,296 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 683,985
Fixed-effects: author_id: 27,091,  year: 75
Standard-errors: IID 
                                 Estimate Std. Error    z value   Pr(>|z|)    
fac_pubBSF                       0.055030   0.012184   4.516442 6.2887e-06 ***
num_author_log                   0.434828   0.008819  49.308389  < 2.2e-16 ***
num_inst_log                     0.059433   0.011308   5.255762 1.4741e-07 ***
num_country_log                 -0.209936   0.017055 -12.309423  < 2.2e-16 ***
num_reference_log                0.272756   0.006136  44.452298  < 2.2e-16 ***
leadership_global_classallNorth  0.060645   0.020574   2.947699 3.2015e-03 ** 
leadership_global_classallSouth -0.012318   0.032691  -0.376790 7.0633e-01    
mean_career_age_log              0.122701   0.009507  12.906676  < 2.2e-16 ***
inst_h_index_log                -0.084335   0.007486 -11.265917  < 2.2e-16 ***
source_h_index_log              -0.095956   0.004236 -22.651096  < 

In [18]:
MEs = ggemmeans(model_ls, terms=c('fac_pub'), typical='median')

In [19]:
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.5368261,0.06506337,0.5050123,0.5683429,1
BSF,0.5504777,0.06521243,0.5186881,0.5818604,1


In [20]:
fname = paste0(main_path, "UseBSForNot/predict_ns_1211_ls.csv")
write.csv(MEs, fname, row.names = FALSE)

In [21]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_hs <- feglm(fml, data = data[(data$domain_Health.Sciences == 1), ], family = binomial("logit"), vcov = "iid")
summary(model_hs)

Warning message in formula.character(object, env = baseenv()):
“Using formula(x) is deprecated when x is a character vector of length > 1.
  Consider formula(paste(x, collapse = " ")) instead.”
NOTES: 1 observation removed because of NA values (RHS: 1).
       16,494/0 fixed-effects (37,598 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 337,939
Fixed-effects: author_id: 18,925,  year: 75
Standard-errors: IID 
                                 Estimate Std. Error   z value   Pr(>|z|)    
fac_pubBSF                       0.191439   0.018946 10.104461  < 2.2e-16 ***
num_author_log                   0.402583   0.012315 32.691391  < 2.2e-16 ***
num_inst_log                     0.065305   0.015225  4.289466 1.7910e-05 ***
num_country_log                 -0.195855   0.023574 -8.308164  < 2.2e-16 ***
num_reference_log                0.447062   0.008820 50.685642  < 2.2e-16 ***
leadership_global_classallNorth  0.116812   0.029545  3.953755 7.6934e-05 ***
leadership_global_classallSouth  0.082914   0.045610  1.817886 6.9082e-02 .  
mean_career_age_log              0.108324   0.014105  7.679601 1.5958e-14 ***
inst_h_index_log                -0.037008   0.010155 -3.644111 2.6832e-04 ***
source_h_index_log              -0.061526   0.006154 -9.998024  < 2.2e-16 ***

In [22]:
MEs = ggemmeans(model_hs, terms=c('fac_pub'), typical='median')
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.5773487,0.08971947,0.5339606,0.6195752,1
BSF,0.6232435,0.09028221,0.5808803,0.6638030,1


In [23]:
fname = paste0(main_path, "UseBSForNot/predict_ns_1211_hs.csv")
write.csv(MEs, fname, row.names = FALSE)

In [24]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_ss <- feglm(fml, data = data[(data$domain_Social.Sciences == 1), ], family = binomial("logit"), vcov = "iid")
summary(model_ss)

Warning message in formula.character(object, env = baseenv()):
“Using formula(x) is deprecated when x is a character vector of length > 1.
  Consider formula(paste(x, collapse = " ")) instead.”
NOTES: 1 observation removed because of NA values (RHS: 1).
       9,528/6 fixed-effects (14,305 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 26,000
Fixed-effects: author_id: 3,308,  year: 68
Standard-errors: IID 
                                 Estimate Std. Error   z value   Pr(>|z|)    
fac_pubBSF                       0.590227   0.091281  6.466058 1.0059e-10 ***
num_author_log                   0.377230   0.054437  6.929618 4.2198e-12 ***
num_inst_log                    -0.057527   0.066008 -0.871509 3.8348e-01    
num_country_log                 -0.382890   0.091946 -4.164269 3.1235e-05 ***
num_reference_log                0.282975   0.030407  9.306158  < 2.2e-16 ***
leadership_global_classallNorth  0.075818   0.092977  0.815451 4.1481e-01    
leadership_global_classallSouth -0.082363   0.128364 -0.641639 5.2111e-01    
mean_career_age_log              0.006393   0.055139  0.115937 9.0770e-01    
inst_h_index_log                 0.013348   0.034633  0.385420 6.9993e-01    
source_h_index_log               0.102373   0.022843  4.481582 7.4092e-06 ***
c

In [25]:
MEs = ggemmeans(model_ss, terms=c('fac_pub'), typical='median')
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.3008051,0.3163430,0.1879345,0.4443698,1
BSF,0.4370263,0.3228685,0.2919256,0.5937696,1


In [26]:
fname = paste0(main_path, "UseBSForNot/predict_ns_1211_ss.csv")
write.csv(MEs, fname, row.names = FALSE)

In [27]:
model1 <- model_ps
model2 <- model_ls
model3 <- model_hs
model4 <- model_ss

country_level_reg <- etable(
    list(model1, model2, model3, model4),
    keep = c("fac_pub", paper_vars, journal_vars ),
    se = "iid",
    # cluster = ~ paper_year + paper_field,
    tex = TRUE,
    digits = 3
)

# writeLines(country_level_reg, con = "selfpromt_reg_1014.tex")

In [28]:
country_level_reg

\begingroup
\centering
\begin{tabular}{lcccc}
   \tabularnewline \midrule \midrule
   Dependent Variable: & \multicolumn{4}{c}{novelbin}\\
   Model:                              & (1)            & (2)            & (3)            & (4)\\  
   \midrule
   \emph{Variables}\\
   fac\_pubBSF                         & 0.075$^{***}$  & 0.055$^{***}$  & 0.191$^{***}$  & 0.590$^{***}$\\   
                                       & (0.007)        & (0.012)        & (0.019)        & (0.091)\\   
   num\_author\_log                    & 0.212$^{***}$  & 0.435$^{***}$  & 0.403$^{***}$  & 0.377$^{***}$\\   
                                       & (0.005)        & (0.009)        & (0.012)        & (0.054)\\   
   num\_inst\_log                      & -0.004         & 0.059$^{***}$  & 0.065$^{***}$  & -0.058\\   
                                       & (0.007)        & (0.011)        & (0.015)        & (0.066)\\   
   num\_country\_log                   & -0.293$^{***}$ & -0.210$^{***}$ & -0.196$^{**